# Preview Tiny Aya JP/EN eval

Loads `answercarefully_eval.sqlite3` written by `eval/eval_jp_en_variants.ipynb` and summarizes how Tiny Aya's Japanese and English answers differ across prompt variants.

On Colab the file is `My Drive/ez-penta/answercarefully_eval.sqlite3`. On a local kernel, download that file from Drive and set `LOCAL_DIR` to the folder that contains it. `pip install google` does not mount Drive: that package is a search scraper, and `google.colab` exists only inside a Colab kernel.

The eval db has one row per answer (`answers`), one row per Japanese/English pair (`pairs`), the summary tables the eval notebook saved, and `eval_info` for the run. This notebook recomputes the rates from `answers` and `pairs` so a snapshot can be inspected on its own.

No GPU needed. On Colab: **Select Kernel → Colab → New Colab Server** (CPU is fine).

In [1]:
import tempfile
from pathlib import Path

DB_NAME = "answercarefully_eval.sqlite3"
# Local kernel: set this to the folder you downloaded the eval db into. Leave None on Colab.
LOCAL_DIR = None

COLAB_DIR = Path("/content/drive/MyDrive/ez-penta")
if LOCAL_DIR:
    DRIVE_DIR = Path(LOCAL_DIR)
elif COLAB_DIR.exists() or Path("/content").exists():
    DRIVE_DIR = COLAB_DIR
else:
    DRIVE_DIR = Path("eval/view") if Path("eval/view").is_dir() else Path(".")

DB = DRIVE_DIR / DB_NAME
WORK_DIR = Path("/content/work") if Path("/content").exists() else Path(tempfile.gettempdir()) / "ez-penta-eval-preview"

VARIANTS = ["original", "fc", "rp", "rs", "every10_yes", "every10_hai", "quartile_yes", "quartile_hai"]
LANGUAGES = ("ja", "en")

## 1. Copy the eval db from Google Drive

In [ ]:
import shutil

if Path("/content").exists() and not COLAB_DIR.exists():
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = COLAB_DIR
    DB = DRIVE_DIR / DB_NAME

if not DB.exists():
    raise FileNotFoundError(
        f"{DB_NAME} was not found in {DRIVE_DIR}.\n"
        "On Colab, run eval/eval_jp_en_variants.ipynb first so it writes My Drive/ez-penta/"
        f"{DB_NAME}. A stage snapshot is answercarefully_eval_<N>rows.sqlite3; set DB_NAME to that file.\n"
        "On this Mac, Google Drive is not mounted and google.colab is not installed. "
        "Download the sqlite file from Drive, set LOCAL_DIR to that folder, and rerun from the top."
    )

available = sorted(path.name for path in DRIVE_DIR.glob("answercarefully_eval*.sqlite3"))
print("Eval databases:")
for name in available:
    print(f"  {name}")

# SQLite is unreliable directly on a Drive mount, so read a local copy.
WORK_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DB = WORK_DIR / DB.name
shutil.copy(DB, LOCAL_DB)
print(f"Copied {DB} -> {LOCAL_DB}")


## 2. Load the tables

In [4]:
import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import binomtest

connection = sqlite3.connect(LOCAL_DB)
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name", connection)["name"]
answers = pd.read_sql_query("SELECT * FROM answers", connection)
pairs = pd.read_sql_query("SELECT * FROM pairs", connection)
eval_info = pd.read_sql_query("SELECT * FROM eval_info", connection)
connection.close()

for column in ("unsafe", "refused", "prompt_flagged", "japanese_answer", "chrf_vs_reference", "similarity_to_reference"):
    answers[column] = pd.to_numeric(answers[column], errors="coerce")
answers["truncated"] = answers["truncated"].map({1: True, 0: False, "1": True, "0": False, True: True, False: False})
for column in ("unsafe_ja", "unsafe_en", "refused_ja", "refused_en", "japanese_answer_ja", "japanese_answer_en", "similarity_ja_en"):
    if column in pairs:
        pairs[column] = pd.to_numeric(pairs[column], errors="coerce")

answers["variant"] = pd.Categorical(answers["variant"], VARIANTS, ordered=True)
pairs["variant"] = pd.Categorical(pairs["variant"], VARIANTS, ordered=True)

print(f"tables: {', '.join(tables)}")
print(f"answers: {len(answers)}  pairs: {len(pairs)}  "
      f"prompts: {answers.groupby(['source_id', 'source_split']).ngroups}  "
      f"variants: {answers['variant'].nunique()}  languages: {sorted(answers['language'].unique())}")
eval_info.T.rename(columns={0: "value"})

ModuleNotFoundError: No module named 'scipy'

## 3. Coverage

`answered` is a non-empty generation. `truncated` hit the token cap and was left unscored. `judged` has a Safe/Unsafe label from Qwen3Guard.

In [ ]:
coverage = answers.groupby(["variant", "language"], observed=True).agg(
    rows=("answer", "size"),
    answered=("answer", lambda column: int(column.notna().sum())),
    truncated=("truncated", lambda column: int((column == True).sum())),  # noqa: E712
    judged=("unsafe", "count"),
    refusal_labels=("refused", "count"),
)
coverage

## 4. How Aya behaves, by variant and language

Rates are means over judged answers. `chrf_vs_reference` is only computed for Japanese answers. `similarity_to_reference` is cosine similarity to the Japanese reference answer.

In [ ]:
by_variant_language = answers.groupby(["variant", "language"], observed=True).agg(
    judged=("unsafe", "count"),
    unsafe_rate=("unsafe", "mean"),
    refusal_rate=("refused", "mean"),
    prompt_flagged_rate=("prompt_flagged", "mean"),
    japanese_answer_rate=("japanese_answer", "mean"),
    chrf_vs_reference=("chrf_vs_reference", "mean"),
    similarity_to_reference=("similarity_to_reference", "mean"),
).round(3)
by_variant_language

In [ ]:
# Change against the original prompt in the same language.
rates = by_variant_language[["unsafe_rate", "refusal_rate", "japanese_answer_rate", "similarity_to_reference"]].unstack("language")
change_vs_original = (rates - rates.loc["original"]).round(3)
change_vs_original.columns = [f"{name}_vs_original_{language}" for name, language in change_vs_original.columns]
change_vs_original

In [ ]:
plot_rates = by_variant_language["refusal_rate"].unstack("language").reindex(VARIANTS)
axes = plot_rates.plot.bar(figsize=(9, 3.5), rot=30, color=["#c45c26", "#3d6b99"])
axes.set_ylim(0, 1)
axes.set_ylabel("refusal rate")
axes.set_title("Refusal rate by variant")
axes.legend(title="answer language")
plt.tight_layout()
plt.show()

## 5. Japanese vs English on the same prompt

A pair is one prompt variant answered in both languages. `ja_only` is the share where the Japanese answer has the outcome and the English answer does not. `mcnemar_p` is an exact test that the two directions are equally likely. `similarity_ja_en` is cosine similarity between the two answers (−1 to 1).

In [ ]:
def mismatch(group, name):
    """Pairs where both languages have a verdict."""
    known = group[[f"{name}_ja", f"{name}_en"]].dropna()
    ja = known[f"{name}_ja"] == 1
    en = known[f"{name}_en"] == 1
    only_ja, only_en, n = int((ja & ~en).sum()), int((~ja & en).sum()), len(known)
    both = int((ja & en).sum())
    neither = int((~ja & ~en).sum())
    return pd.Series({
        f"{name}_pairs": n,
        f"{name}_both": both / n if n else np.nan,
        f"{name}_neither": neither / n if n else np.nan,
        f"{name}_ja_only": only_ja / n if n else np.nan,
        f"{name}_en_only": only_en / n if n else np.nan,
        f"{name}_mismatch_rate": (only_ja + only_en) / n if n else np.nan,
        f"{name}_mcnemar_p": binomtest(only_ja, only_ja + only_en, 0.5).pvalue if only_ja + only_en else np.nan,
    })


similarity = pairs.groupby("variant", observed=True)["similarity_ja_en"].agg(
    similarity_ja_en="mean", similarity_ja_en_median="median")
by_variant = pd.concat([
    pairs.groupby("variant", observed=True).apply(mismatch, "refused", include_groups=False),
    pairs.groupby("variant", observed=True).apply(mismatch, "unsafe", include_groups=False),
    similarity,
], axis=1)
by_variant.reindex(VARIANTS).round(3)

In [ ]:
direction = by_variant.reindex(VARIANTS)[["refused_ja_only", "refused_en_only"]]
axes = direction.plot.bar(stacked=True, figsize=(9, 3.5), rot=30, color=["#c45c26", "#3d6b99"])
axes.set_ylabel("share of pairs")
axes.set_title("Refusal mismatch direction")
axes.legend(["Japanese refused, English complied", "English refused, Japanese complied"])
plt.tight_layout()
plt.show()

When the two answers disagree on refusal, are they also saying different things? `similarity_ja_en` split by whether the refusal labels match.

In [ ]:
labeled = pairs.dropna(subset=["refused_ja", "refused_en", "similarity_ja_en"]).copy()
labeled["refusal_agreement"] = np.where(labeled["refused_ja"] == labeled["refused_en"], "same refusal", "refusal mismatch")
labeled.groupby(["variant", "refusal_agreement"], observed=True)["similarity_ja_en"].agg(["count", "mean", "median"]).round(3)

## 6. Where the languages disagree

Refusal-mismatch rate by AnswerCarefully risk area and harm type. Each cell is the share of labeled pairs in that bucket.

In [ ]:
known = pairs.dropna(subset=["refused_ja", "refused_en"]).copy()
known["refusal_mismatch"] = (known["refused_ja"] != known["refused_en"]).astype(float)
known["ja_refused_only"] = ((known["refused_ja"] == 1) & (known["refused_en"] == 0)).astype(float)

by_risk = known.pivot_table(index="risk_area_en", columns="variant", values="refusal_mismatch", aggfunc="mean", observed=True)
counts = known.groupby("risk_area_en").size().rename("pairs")
by_risk = by_risk.reindex(columns=VARIANTS)
by_risk.insert(0, "pairs", counts)
by_risk.sort_values("original", ascending=False).round(2)

In [ ]:
by_harm = known.pivot_table(index="harm_type_en", columns="variant", values="refusal_mismatch", aggfunc="mean", observed=True)
harm_counts = known.groupby("harm_type_en").size().rename("pairs")
by_harm = by_harm.reindex(columns=VARIANTS)
by_harm.insert(0, "pairs", harm_counts)
by_harm.sort_values("original", ascending=False).round(2)

In [ ]:
# Among mismatches, how often it is the Japanese answer that refuses.
mismatches = known[known["refusal_mismatch"] == 1]
direction_by_risk = mismatches.groupby("risk_area_en").agg(
    mismatches=("ja_refused_only", "size"),
    ja_only_share=("ja_refused_only", "mean"),
).sort_values("mismatches", ascending=False).round(2)
direction_by_risk

## 7. Answer language and prompt flags

`japanese_answer` is a run of 8+ kana or kanji. A Japanese prompt should land near 1 and an English prompt near 0. `prompt_flagged` is Qwen3Guard marking the prompt itself Unsafe.

In [ ]:
answers.pivot_table(index="variant", columns="language", values="japanese_answer", aggfunc="mean", observed=True).reindex(VARIANTS).round(3)

In [ ]:
flagged = answers.dropna(subset=["prompt_flagged", "refused"]).copy()
flagged["prompt"] = np.where(flagged["prompt_flagged"] == 1, "prompt flagged", "prompt clear")
flagged.groupby(["language", "prompt"], observed=True).agg(
    answers=("refused", "size"),
    refusal_rate=("refused", "mean"),
    unsafe_rate=("unsafe", "mean"),
).round(3)

## 8. Guard categories

Qwen3Guard category names on answers it judged Unsafe, counted once per answer.

In [ ]:
unsafe_answers = answers[answers["unsafe"] == 1].copy()
unsafe_answers["category"] = unsafe_answers["guard_categories"].fillna("(none)").str.split(r",\s*")
categories = unsafe_answers.explode("category")
categories["category"] = categories["category"].replace("", "(none)")
top = categories.groupby(["language", "category"]).size().rename("answers").reset_index()
top.sort_values(["language", "answers"], ascending=[True, False]).groupby("language").head(8)

## 9. Example mismatches

Change `VARIANT` to look at another prompt variant. Each row is a pair whose refusal labels disagree.

In [ ]:
VARIANT = "original"
pd.set_option("display.max_colwidth", 400)

answer_text = answers.set_index(["source_id", "source_split", "variant", "language"])
examples = pairs[(pairs["variant"] == VARIANT) & (pairs["refused_ja"] != pairs["refused_en"])
                 & pairs["refused_ja"].notna() & pairs["refused_en"].notna()].sort_values("similarity_ja_en").head(8)


def lookup(row, language, column):
    return answer_text.at[(row.source_id, row.source_split, VARIANT, language), column]


pd.DataFrame([{
    "source_id": row.source_id,
    "risk_area_en": row.risk_area_en,
    "refused_ja": row.refused_ja,
    "refused_en": row.refused_en,
    "similarity_ja_en": round(row.similarity_ja_en, 3) if pd.notna(row.similarity_ja_en) else None,
    "prompt_ja": lookup(row, "ja", "prompt"),
    "answer_ja": lookup(row, "ja", "answer"),
    "answer_en": lookup(row, "en", "answer"),
} for row in examples.itertuples(index=False)])

## 10. Do dataset biases explain the Japanese/English gap?

Joins `answercarefully_bias_features.sqlite3` (saved by section 9 of `preview_prompts.ipynb`) onto the pairs. Each table splits one outcome by one dataset feature. If the refusal mismatch is concentrated in items adapted for Japan, or in one prompt form, part of the language gap is that feature.

In [ ]:
FEATURES_NAME = "answercarefully_bias_features.sqlite3"
if not (DRIVE_DIR / FEATURES_NAME).exists():
    raise FileNotFoundError(f"{FEATURES_NAME} is not in {DRIVE_DIR}. Run section 9 of eval/view/preview_prompts.ipynb first.")
shutil.copy(DRIVE_DIR / FEATURES_NAME, WORK_DIR / FEATURES_NAME)
connection = sqlite3.connect(WORK_DIR / FEATURES_NAME)
features = pd.read_sql_query("SELECT * FROM features", connection)
connection.close()

BIAS_FEATURES = ["culture", "adaptation_tag", "mentions_japan", "prompt_form", "first_person", "length_band", "reference_declines"]
KEY = ["source_id", "source_split"]
feature_pairs = pairs.merge(features[KEY + BIAS_FEATURES], on=KEY, how="left", validate="many_to_one")
feature_answers = answers.merge(features[KEY + BIAS_FEATURES], on=KEY, how="left", validate="many_to_one")
print(f"Features joined to {feature_pairs[BIAS_FEATURES[0]].notna().sum()} of {len(feature_pairs)} pairs")

In [ ]:
# Refusal and safety by feature, original prompts. Change BIAS_VARIANT to look at another variant.
BIAS_VARIANT = "original"


def split_by(frame, feature):
    rows = []
    for level, group in frame.groupby(feature, observed=True):
        refused, unsafe = mismatch(group, "refused"), mismatch(group, "unsafe")
        rows.append({
            "feature": feature, "level": level, "pairs": refused["refused_pairs"],
            "refused_ja": group["refused_ja"].mean(), "refused_en": group["refused_en"].mean(),
            "refusal_mismatch": refused["refused_mismatch_rate"], "ja_only": refused["refused_ja_only"],
            "en_only": refused["refused_en_only"], "refusal_mcnemar_p": refused["refused_mcnemar_p"],
            "unsafe_ja": group["unsafe_ja"].mean(), "unsafe_en": group["unsafe_en"].mean(),
            "unsafe_mismatch": unsafe["unsafe_mismatch_rate"],
        })
    return pd.DataFrame(rows)


chosen = feature_pairs[feature_pairs["variant"] == BIAS_VARIANT]
pd.concat([split_by(chosen, feature) for feature in BIAS_FEATURES], ignore_index=True).set_index(["feature", "level"]).round(3)

In [ ]:
# Refusal mismatch by culture, for every variant.
known_bias = feature_pairs.dropna(subset=["refused_ja", "refused_en"]).copy()
known_bias["refusal_mismatch"] = (known_bias["refused_ja"] != known_bias["refused_en"]).astype(float)
display(known_bias.pivot_table(index="variant", columns="culture", values="refusal_mismatch", aggfunc="mean", observed=True)
        .reindex(VARIANTS).round(3))

# Within each risk area: does adaptation still matter? (original prompts)
original_known = known_bias[known_bias["variant"] == "original"]
area_culture = original_known.pivot_table(index="risk_area_en", columns="culture", values="refusal_mismatch", aggfunc="mean", observed=True)
area_culture["pairs"] = original_known.groupby("risk_area_en").size()
area_culture.round(3)

### 10.1 Is the guard biased by language?

The Japanese and English original prompts ask the same thing, so Qwen3Guard should flag them at the same rate. `flagged_ja_only` and `flagged_en_only` are prompts the guard marked Unsafe in one language only. A large one-sided share means the judge itself, not Aya, moves with language.

In [ ]:
flags = feature_answers[feature_answers["variant"] == "original"].pivot_table(
    index=KEY + ["culture"], columns="language", values="prompt_flagged", aggfunc="first", observed=True)
flags.columns = [f"prompt_flagged_{language}" for language in flags.columns]
flags = flags.reset_index()


def flag_gap(group):
    known = group[["prompt_flagged_ja", "prompt_flagged_en"]].dropna()
    ja, en = known["prompt_flagged_ja"] == 1, known["prompt_flagged_en"] == 1
    only_ja, only_en = int((ja & ~en).sum()), int((~ja & en).sum())
    return pd.Series({
        "pairs": len(known), "flagged_ja": ja.mean(), "flagged_en": en.mean(),
        "flagged_ja_only": only_ja / len(known) if len(known) else np.nan,
        "flagged_en_only": only_en / len(known) if len(known) else np.nan,
        "mcnemar_p": binomtest(only_ja, only_ja + only_en, 0.5).pvalue if only_ja + only_en else np.nan,
    })


if {"prompt_flagged_ja", "prompt_flagged_en"} <= set(flags.columns):
    display(pd.concat({"all": flag_gap(flags)}, axis=1).T.round(3))
    display(flags.groupby("culture").apply(flag_gap, include_groups=False).round(3))
else:
    print("No prompt flags in both languages yet.")

### 10.2 Refusal against the reference answer

`reference_declines` is whether the Japanese reference answer declines. Agreement is Aya refusing exactly when the reference declines. `refused_but_reference_answers` is over-refusal against the reference, and `answered_but_reference_declines` is under-refusal. The reference flag is a phrase match, so read these as estimates.

In [ ]:
def reference_agreement(group):
    known = group.dropna(subset=["refused", "reference_declines"])
    refused = known["refused"] == 1
    declines = known["reference_declines"] == 1
    return pd.Series({
        "answers": len(known),
        "agreement": (refused == declines).mean() if len(known) else np.nan,
        "refused_but_reference_answers": (refused & ~declines).mean() if len(known) else np.nan,
        "answered_but_reference_declines": (~refused & declines).mean() if len(known) else np.nan,
    })


feature_answers.groupby(["variant", "language"], observed=True).apply(reference_agreement, include_groups=False).reindex(VARIANTS, level=0).round(3)